In [22]:
import os
import openml
import pandas as pd
from tqdm import tqdm
from openml import config
import numpy as np
from collections import defaultdict
import random
import json
from sklearn.preprocessing import LabelEncoder

# Set your OpenML API key
config.apikey = 'c0d6200b271e73a8aec0904980876c3c'

### Read mappings

In [23]:
with open("flows/filtered_flow_algorithm_mapping_v2.json", "r") as f:
    flow_map = json.load(f)

valid_flow_ids = set(map(int, flow_map.keys()))

### define tasks and suites

In [24]:
suite_id = 225
single_task_id = 58
benchmark_suite = openml.study.get_suite(suite_id)
task_ids = benchmark_suite.tasks

### Single task runs

In [25]:
runs_df = openml.runs.list_runs(task=[single_task_id], output_format='dataframe')
runs_df.set_index('run_id', inplace=True)

raw_dir = os.path.join("runs", "raw")
os.makedirs(raw_dir, exist_ok=True)


output_path = os.path.join(raw_dir, f"task_{single_task_id}_runs.csv")
runs_df.to_csv(output_path)

print(f"Saved {len(runs_df)} runs to {output_path}")


Saved 14598 runs to runs/raw/task_58_runs.csv


### Filter to match

In [26]:
filtered_runs_df = runs_df[runs_df['flow_id'].isin(valid_flow_ids)]

# === Print filtering summary ===
print(f"Total runs before filtering: {len(runs_df)}")
print(f"Total runs after filtering: {len(filtered_runs_df)}")
print(f"Reduction: {(len(runs_df) - len(filtered_runs_df)) / len(runs_df):.2%}")

# === Save to 'runs/filtered' folder ===
filtered_dir = os.path.join("runs", "filtered")
os.makedirs(filtered_dir, exist_ok=True)

output_path = os.path.join(filtered_dir, f"task_{single_task_id}_runs.csv")
filtered_runs_df.to_csv(output_path)

print(f"Filtered runs saved to {output_path}")

Total runs before filtering: 14598
Total runs after filtering: 10369
Reduction: 28.97%
Filtered runs saved to runs/filtered/task_58_runs.csv


### Group flow_ids according to algorithm types

In [27]:
algo_to_flows = defaultdict(list)

for flow_id_str, details in flow_map.items():
    algo_type = details.get("algorithm_type", "Unknown")
    flow_id = int(flow_id_str)  
    algo_to_flows[algo_type].append(flow_id)

### Split by algorithm

In [28]:
filtered_csv_path = os.path.join("runs", "filtered", f"task_{single_task_id}_runs.csv")

runs_df = pd.read_csv(filtered_csv_path)
runs_df.set_index("run_id", inplace=True)


flow_to_algorithm = {
    int(fid): entry["algorithm_type"]
    for fid, entry in flow_map.items()
}


task_output_dir = os.path.join("runs", "algorithm_splits", f"task_{single_task_id}")
os.makedirs(task_output_dir, exist_ok=True)


alg_groups = defaultdict(list)

for run_id, row in runs_df.iterrows():
    flow_id = row["flow_id"]
    algo_type = flow_to_algorithm.get(flow_id)
    if algo_type:
        alg_groups[algo_type].append(run_id)


for algo_type, run_ids in alg_groups.items():
    filename = f"{algo_type.lower().replace(' ', '_')}_runs.csv"
    output_path = os.path.join(task_output_dir, filename)
    runs_df.loc[run_ids].to_csv(output_path)
    print(f"Saved {len(run_ids)} runs to {output_path}")


Saved 2456 runs to runs/algorithm_splits/task_58/decision_tree_runs.csv
Saved 2320 runs to runs/algorithm_splits/task_58/random_forest_runs.csv
Saved 5353 runs to runs/algorithm_splits/task_58/support_vector_machine_runs.csv
Saved 240 runs to runs/algorithm_splits/task_58/xgboost_runs.csv


### Accuracy for each split

In [29]:
def chunks(lst, n):
    """Yield successive n-sized chunks from lst."""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]


task_output_dir = os.path.join("runs", "algorithm_splits", f"task_{single_task_id}")
BATCH_SIZE = 400


for filename in os.listdir(task_output_dir):
    if filename.endswith(".csv"):
        csv_path = os.path.join(task_output_dir, filename)
        print(f"\nProcessing: {filename}")

        # Load CSV and set run_id as index
        df = pd.read_csv(csv_path)
        if 'run_id' not in df.columns:
            print(f"Skipping {filename}: no run_id column.")
            continue
        df.set_index("run_id", inplace=True)

        if df.empty:
            print(f"No runs to process in {filename}.")
            continue

        run_ids = df.index.tolist()
        print(f"Fetching evaluations for {len(run_ids)} runs...")

        accuracy_map = {}

        
        for batch in tqdm(list(chunks(run_ids, BATCH_SIZE)), desc="Batch fetching", leave=False):
            try:
                evaluations = openml.evaluations.list_evaluations(
                    function='predictive_accuracy',
                    runs=[int(rid) for rid in batch],
                    output_format='dataframe'
                )
                if not evaluations.empty:
                    accuracy_map.update(dict(zip(evaluations['run_id'], evaluations['value'])))
            except Exception as e:
                print(f"⚠️ Error in batch {batch[0]}–{batch[-1]}: {e}")
                continue

        
        df['predictive_accuracy'] = df.index.map(lambda x: accuracy_map.get(int(x), None))

        
        df.to_csv(csv_path)
        print(f"Updated {filename} with predictive accuracy.")



Processing: random_forest_runs.csv
Fetching evaluations for 2320 runs...


Updated random_forest_runs.csv with predictive accuracy.

Processing: xgboost_runs.csv
Fetching evaluations for 240 runs...


Updated xgboost_runs.csv with predictive accuracy.

Processing: support_vector_machine_runs.csv
Fetching evaluations for 5353 runs...


Updated support_vector_machine_runs.csv with predictive accuracy.

Processing: decision_tree_runs.csv
Fetching evaluations for 2456 runs...


Updated decision_tree_runs.csv with predictive accuracy.


### Statistics

In [30]:
task_output_dir = os.path.join("runs", "algorithm_splits", f"task_{single_task_id}")
summary_path = os.path.join(task_output_dir, "accuracy_summary.csv")


summary_rows = []

for filename in os.listdir(task_output_dir):
    if filename.endswith(".csv"):
        csv_path = os.path.join(task_output_dir, filename)
        df = pd.read_csv(csv_path)

        if 'predictive_accuracy' not in df.columns or df['predictive_accuracy'].dropna().empty:
            print(f"Skipping {filename} — no predictive accuracy.")
            continue

        accs = df['predictive_accuracy'].dropna()

        stats = {
            'algorithm': filename.replace("_runs.csv", "").replace("_", " ").title(),
            'count': len(accs),
            'mean': accs.mean(),
            'std': accs.std(),
            'min': accs.min(),
            '25%': accs.quantile(0.25),
            '50% (median)': accs.median(),
            '75%': accs.quantile(0.75),
            'max': accs.max(),
            'top10_median': accs.sort_values(ascending=False).head(10).median()

        }

        summary_rows.append(stats)

# Create and save summary DataFrame
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(summary_path, index=False)
print(f"Saved summary statistics to {summary_path}")
summary_df


Saved summary statistics to runs/algorithm_splits/task_58/accuracy_summary.csv


,algorithm,count,mean,std,min,25%,50% (median),75%,max,top10_median
0,Random Forest,2320,0.801763,0.128040,0.3384,0.82940,0.8398,0.84680,0.8694,0.8585
1,Xgboost,240,0.529303,0.232667,0.3300,0.33325,0.3384,0.81035,0.8604,0.8565
2,Support Vector Machine,5353,0.699935,0.230422,0.0000,0.60440,0.8288,0.86140,0.8698,0.8687
3,Decision Tree,2456,0.639211,0.179538,0.3384,0.56860,0.7508,0.76120,0.8702,0.8702
